In [ ]:
"""
device = True if torch.cuda.is_available() else False
num_epoch = 500
generator_losses = []
discriminator_losses = []
for epoch in range(num_epoch):
    for batch in dataloader:
        model = CTGAN(epochs=num_epoch, batch_size=500, generator_dim=(256, 256, 256), discriminator_dim=(256, 256, 256), verbose=True, cuda=device)
        model.fit(data, discrete_columns=['SID'])
"""

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm

In [ ]:
print("PyTorch: ", torch.__version__)
print("CUDA: ", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print("Numpy: ", np.__version__)

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
# ====Mode-specific normalization====
def normalize_mode_specific(df, discrete_col, continuous_cols):
    grouped = df.groupby(discrete_col)
    mean_df = grouped[continuous_cols].transform('mean')
    std_df = grouped[continuous_cols].transform('std').replace(0, 1e-6)

    df_norm = df.copy()
    df_norm[continuous_cols] = (df[continuous_cols] - mean_df) / std_df

    mean_dict = grouped[continuous_cols].mean().to_dict()
    std_dict = grouped[continuous_cols].std().replace(0, 1e-6).to_dict()

    return df_norm, mean_dict, std_dict

def denormalize_generated(generated_array, sid_array, continuous_cols, mean_dict, std_dict):
    output = []
    for row, sid in zip(generated_array, sid_array):
        denorm_row = []
        for val, col in zip(row, continuous_cols):
            mean = mean_dict[col].get(sid, 0)
            std = std_dict[col].get(sid, 1)
            denorm_row.append(val * std + mean)
        output.append(denorm_row)
    return np.array(output)
    

def create_conditional_vector(class_indices, num_classes):
    return torch.nn.functional.one_hot(class_indices, num_classes=num_classes).float()

In [ ]:
# ====== Configuration ======
pac = 2                      # PAC size
batch_size = 250
num_epochs = 100
z_dim = 128
embed_dim = 16
d_lr = 2e-4
g_lr = 2e-4
l2_decay = 1e-5 # 1e-5: small, 1e-4: moderate, 1e-3: stronger

# =====Load & Preprocess=====
data_path = r"D:/ForestFire/CBH/data/NFI6-7_cleaned2.csv"
drop_columns = ['SampleID', 'Cycle', 'I_Species', 'Species']
condition_col = 'SID'        # Use SID as the conditional vector

df = pd.read_csv(data_path, encoding='cp949')
df = df.drop(columns=drop_columns)

le_sid = LabelEncoder()
le_imsang = LabelEncoder()
df['SID'] = le_sid.fit_transform(df['SID'])
df['Imsang'] = le_imsang.fit_transform(df['Imsang'])

continuous_cols = df.columns.difference(['SID', 'Imsang']).tolist()
df, mean_dict, std_dict = normalize_mode_specific(df, 'SID', continuous_cols)

# Convert to tensors
real_data_tensor = torch.tensor(df[continuous_cols].values, dtype=torch.float32)
sid_tensor = torch.tensor(df['SID'].values, dtype=torch.long)
imsang_tensor = torch.tensor(df['Imsang'].values, dtype=torch.long)
dataset = DataLoader(TensorDataset(real_data_tensor, sid_tensor, imsang_tensor), batch_size=batch_size, shuffle=True)

sid_num_classes = sid_num_classes = len(torch.unique(sid_tensor))
real_dim = len(continuous_cols)

assert sid_tensor.max().item() < sid_num_classes, "SID index out of embedding range"

In [ ]:
# ====== Generator ======
class Generator(nn.Module):
    # 모델 함수
    def __init__(self, z_dim, embed_dim, output_dim, sid_classes):
        """
        z_dim: input tensor의 feature column 개수
        embed_dim: condition tensor의 차원
        sid_classess: 모델에서 학습 시 고려할 categorical feature의 클래스 수
        """
        super().__init__()
        self.sid_emb = nn.Embedding(sid_classes, embed_dim)
        self.model = nn.Sequential(
            nn.Linear(z_dim + embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    # 모델에 input tensor을 forward하는 함수
    def forward(self, z, sid_idx):
        sid_e = self.sid_emb(sid_idx)
        x = torch.cat([z, sid_e], dim=1)
        return self.model(x)

# ====== Discriminator ======
class Discriminator(nn.Module):
    def __init__(self, input_dim, embed_dim, sid_classes, pac=1):
        super().__init__()
        self.pac = pac
        self.sid_emb = nn.Embedding(sid_classes, embed_dim)
        self.model = nn.Sequential(
            nn.Linear((input_dim + embed_dim) * pac, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x, sid_idx):
        sid_e = self.sid_emb(sid_idx)
        if self.pac > 1:
        # embed_dim * pac 로 flatten 해줘야 concat 시 크기 맞음
            sid_e = sid_e.repeat(1, self.pac)  # [batch//pac, embed_dim * pac]
        # ===== Debugging =====
        """
        print("=== Before reshape ===")
        print("x.shape:", x.shape)         # Expect: [batch, 11]
        print("sid_e.shape:", sid_e.shape) # Expect: [batch, 16]
        
        x = x.view(x.size(0) // self.pac, -1)
        sid_e = sid_e.view(sid_e.size(0) // self.pac, -1)
        
        """
        """
        # ===== Debugging =====
        print("=== After reshape ===")
        print("x.shape:", x.shape)         # Expect: [batch/2, 22]
        print("sid_e.shape:", sid_e.shape) # Expect: [batch/2, 32]
        print("concated.shape:", (torch.cat([x, sid_e], dim=1)).shape)  # Expect: [batch/2, 54]
        """

        return self.model(torch.cat([x, sid_e], dim=1))

# ====== Models & Optimizers ======
device = 'cuda' # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G = Generator(z_dim, embed_dim, real_dim, sid_num_classes).to(device)
print("sid_emb num_embeddings:", G.sid_emb.num_embeddings)
print("sid_tensor max index:", sid_tensor.max().item())
D = Discriminator(real_dim, embed_dim, sid_num_classes, pac=pac).to(device)

print("Embedding weight shape:", G.sid_emb.weight.shape)

loss_fn = nn.BCELoss()
g_optimizer = optim.Adam(G.parameters(), lr=g_lr,betas=(0.5, 0.9),  weight_decay = l2_decay)
d_optimizer = optim.Adam(D.parameters(), lr=d_lr, betas=(0.5, 0.9), weight_decay = l2_decay)

In [ ]:
# ====== Safety Checks Before Model ======
print("SID tensor dtype:", sid_tensor.dtype)
print("SID tensor min:", sid_tensor.min().item())
print("SID tensor max:", sid_tensor.max().item())
print("SID embedding size:", sid_num_classes)

assert sid_tensor.dtype == torch.long, "SID tensor must be torch.long"
assert sid_tensor.min().item() >= 0, "SID contains negative index"
assert sid_tensor.max().item() < sid_num_classes, "SID index exceeds embedding size"

In [ ]:
for i, (real_data, sid_idx, imsang_idx) in enumerate(dataset):
    assert sid_idx.max().item() < sid_num_classes

In [ ]:
training = True
noise_std = 0.1 # std dev of Gaussian noise to add to discriminator input (to lower the capacity of the discriminator)
for epoch in tqdm(range(num_epochs)):
    for i, (real_data, sid_idx, imsang_idx) in enumerate(dataset):
        real_data = real_data.to(device)
        sid_idx = sid_idx.to(device)
        sid_idx_orig = sid_idx.clone()  # ✅ 원본 보관

        batch_len = sid_idx.size(0)
        trimmed_len = (batch_len // pac) * pac

        # ====== Train Discriminator ======
        z = torch.randn(batch_len, z_dim).to(device)
        fake_data = G(z, sid_idx).detach()

        if pac > 1:
            d_real_input = real_data[:trimmed_len]
            d_fake_input = fake_data[:trimmed_len]
            sid_idx_trimmed = sid_idx[:trimmed_len]

            d_real_input = d_real_input.view(trimmed_len // pac, -1)
            d_fake_input = d_fake_input.view(trimmed_len // pac, -1)
            sid_idx_pac = sid_idx_trimmed.view(trimmed_len // pac, pac)[:, 0]
        else:
            d_real_input = real_data
            d_fake_input = fake_data
            sid_idx_pac = sid_idx
         # ===== Improvement-2: Apply Gaussian Noise =====
        if training:
            d_real_input = d_real_input + noise_std * torch.randn_like(d_real_input)
            d_fake_input = d_fake_input + noise_std * torch.randn_like(d_fake_input)
            
        # ===== Imrpovement-1: label smoothing =====
        real_labels = torch.full((d_real_input.size(0), 1), 0.9, device=device)  # Smoothed real labels
        fake_labels = torch.zeros(d_fake_input.size(0), 1).to(device)


        d_real_out = D(d_real_input, sid_idx_pac)
        d_fake_out = D(d_fake_input, sid_idx_pac)
        d_loss = loss_fn(d_real_out, real_labels) + loss_fn(d_fake_out, fake_labels)

        D.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # ====== Train Generator ======
        z = torch.randn(batch_len, z_dim).to(device)
        fake_data = G(z, sid_idx_orig)  # ✅ 원본 사용
        # print('average of generated data: ', fake_data.mean())

        if pac > 1:
            fake_data_trimmed = fake_data[:trimmed_len]
            sid_idx_trimmed = sid_idx_orig[:trimmed_len]
            g_input = fake_data_trimmed.view(trimmed_len // pac, pac * real_dim)
            sid_idx_pac = sid_idx_trimmed.view(trimmed_len // pac, pac)[:, 0]
        else:
            g_input = fake_data
            sid_idx_pac = sid_idx_orig

        g_labels = torch.ones(g_input.size(0), 1).to(device)
        g_loss = -torch.mean(torch.log(D(g_input, sid_idx_pac) + 1e-8)) # g_loss = loss_fn(D(g_input, sid_idx_pac), g_labels)


        G.zero_grad()
        g_loss.backward()
        g_optimizer.step()

        # ====== Logging ======
        log_file.write(f"Epoch {epoch}, Batch {i}, Gen Loss: {g_loss.item():.4f}, Disc Loss: {d_loss.item():.4f}\n")

    # if (epoch + 1) % 50 == 0 or epoch == 0:
    print(f"Epoch {epoch+1}/{num_epochs} | Gen Loss: {g_loss.item():.4f}, Disc Loss: {d_loss.item():.4f}")


In [ ]:
# are weights of the generator updating?
for name, param in G.named_parameters():
    if param.requires_grad and param.grad is not None:
        print(f"{name} grad norm: {param.grad.norm().item():.4f}")

In [ ]:
# visuzlize the loss changes
import pandas as pd
import matplotlib.pyplot as plt
import re

# Load the log file
log_path = "log.txt"

# Read and parse the log file
with open(log_path, "r") as f:
    lines = f.readlines()

# Extract epoch, generator loss, and discriminator loss using regex
log_data = []
pattern = re.compile(r"Epoch (\d+), Batch \d+, Gen Loss: ([\d.]+), Disc Loss: ([\d.]+)")

for line in lines:
    match = pattern.search(line)
    if match:
        epoch = int(match.group(1))
        gen_loss = float(match.group(2))
        disc_loss = float(match.group(3))
        log_data.append((epoch, gen_loss, disc_loss))

# Convert to DataFrame
df_log = pd.DataFrame(log_data, columns=["Epoch", "GenLoss", "DiscLoss"])

# Compute average loss per epoch
df_avg = df_log.groupby("Epoch").mean().reset_index()

# Plot
plt.figure(figsize=(10, 6))
plt.plot(df_avg["Epoch"], df_avg["GenLoss"], label="Generator Loss")
plt.plot(df_avg["Epoch"], df_avg["DiscLoss"], label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Generator & Discriminator Loss Over Epochs")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
for i, (real_data, sid_idx, imsang_idx) in enumerate(dataset):
    print("real_data.shape:", real_data.shape)
    break
print("Columns in df:", df.columns.tolist())
print("Columns in coninous cols:", continuous_cols)
print("real_dim size: ", real_dim)

In [ ]:
# =====Post-train Generation + Save=====
sample_size = 100
z = torch.randn(sample_size, z_dim).to(device) # 정규분포에서 랜덤 숫자를 생성해 반환하는 함수
sid_sample = torch.randint(0, sid_num_classes, (sample_size,), dtype=torch.long).to(device)

fake_norm = G(z, sid_sample).detach().cpu().numpy()
sid_np = sid_sample.cpu().numpy()
fake_denorm = denormalize_generated(fake_norm, sid_np, continuous_cols, mean_dict, std_dict)

# Create DataFrame
df_fake = pd.DataFrame(fake_denorm, columns=continuous_cols)
df_fake['SID'] = le_sid.inverse_transform(sid_np)

# Save if needed
# df_fake.to_csv("generated_data.csv", index=False)
print(df_fake.head())

In [ ]:
# sample 생성하기
samples = model.sample(10000)
samples['Imsang'] = le.inverse_transform(samples['Imsang'])
plt.hist(samples['SID'])

In [ ]:
# 데이터 유효성 평가
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data=df,
    synthetic_data=synthetic_data,
    metadata=metadata)

quality_report.get_details(property_name='Column Shapes')

In [ ]:
# 분포 비교 시각화
from sdv.evaluation.single_table import get_column_plot

fig = get_column_plot(
    real_data=df,
    synthetic_data=synthetic_data,
    metadata=metadata,
    column_name='대출금액'
)

fig.show()

In [ ]:
# ecologically resaonable한지 판단하는 방법 필요